# v11 — satu model tim (notebook yang sama untuk keenam model)

Model ditentukan oleh baris `MODEL` di sel **Setelan**. Panduan lengkap tim: `../README.md`.

**Cara pakai**
1. Ubah hanya sel **Setelan**. `SETELAN` untuk mencoba setelan lain, dan **ganti `NAMA_SETELAN` setiap kali `SETELAN` berubah**, supaya hasil lama tidak tertimpa (kalau lupa, notebook berhenti dengan pesan).
2. *Run All*. Training 5 ulangan × 5 fold pasien (protokol v3) bisa dilanjutkan: kalau sesi Kaggle/Colab terputus, jalankan lagi; fold yang sudah selesai dilewati.
3. Isi sel **Kesimpulan** dengan angka dari tabel, lalu bagikan zip hasil (sel terakhir) ke tim.

`SIHEDAF_QUICK=1` hanya untuk uji alur (data kecil, 1 epoch). Hasilnya **bukan** hasil penelitian dan disimpan terpisah (`cache/v11_quick/`).

In [1]:
# ── Setelan: satu-satunya sel yang perlu diubah ────────────────────────────────
MODEL = "xgboost"        # nama folder model (jangan diubah)
NAMA_SETELAN = "bawaan"    # GANTI setiap kali SETELAN diubah, mis. "percobaan2"
SETELAN = {}               # menimpa setelan bawaan model.py, mis. {"epochs": 40}; kunci yang ada dicetak di bagian 1
#   balancing (opsional, hanya window latih tiap fold): {"balancing": "under"} atau "over" untuk semua model,
#   {"balancing": "smote"} hanya svm/xgboost. Bawaan None = tanpa balancing (lihat ../README.md)
HASIL_DI = None            # Colab: mis. "/content/drive/MyDrive/sihedaf_v11" agar hasil tidak hilang (lihat README)
DATA_DI = None             # isi hanya bila unduhan data gagal: folder hasil ekstrak v11_data.zip (lihat README)
REPO_URL = "https://github.com/farhanlhsn/WrapSihedaf.git"
BRANCH = "main"            # pakai fork/branch sendiri bila mengubah arsitektur di model.py
DATA_URL = "https://github.com/farhanlhsn/WrapSihedaf/releases/download/v11-data/v11_data.zip"

import os
import subprocess
import sys
from pathlib import Path

if HASIL_DI and str(HASIL_DI).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive                      # Colab: sambungkan Google Drive
    drive.mount("/content/drive")
if HASIL_DI:
    os.environ["SIHEDAF_HASIL"] = str(HASIL_DI)
if DATA_DI:
    os.environ["SIHEDAF_DATA"] = str(DATA_DI)
if not Path("model.py").exists():                       # Kaggle/Colab: clone repo dulu
    root = Path("/kaggle/working/WrapSihedaf") if Path("/kaggle/working").exists() else Path.cwd() / "WrapSihedaf"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(root)], check=True)
    os.chdir(root / "v11_model_tim" / MODEL)
    req = Path(f"requirements-{MODEL}.txt")
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
assert Path.cwd().name == MODEL, f"cwd harus folder v11_model_tim/{MODEL}, sekarang {Path.cwd()}"
for p in (str(Path.cwd().parent), str(Path.cwd())):
    if p not in sys.path:
        sys.path.insert(0, p)

from bersama import data, evaluasi, runner

data.siapkan_data(DATA_URL)
import torch

print("cwd   :", Path.cwd())
print("commit:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")
print("data  :", data.lokasi_data())
print("hasil :", data.lokasi_hasil())
print("QUICK :", data.QUICK, "(uji alur, BUKAN hasil)" if data.QUICK else "")

cwd   : /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/v11_model_tim/xgboost
commit: b18d6bc docs: tegaskan hanya paket v3.1 yang siap pakai untuk backend & uji perangkat
GPU   : NVIDIA GeForce RTX 5060
data  : /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache
hasil : /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11
QUICK : False 


## 1. Data dan model

In [2]:
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)   # teks vonis tidak terpotong

import model as MDL

d = data.muat()
print(f"{len(d.y):,} window uji detektor, {len(d.patients)} pasien, fold {d.folds.shape} (ulangan × window)")
print("Setelan bawaan:", {**runner.SETELAN_BERSAMA, **MDL.SETELAN})
print("Ditimpa       :", SETELAN)

165,978 window uji detektor, 23 pasien, fold (5, 165978) (ulangan × window)
Setelan bawaan: {'balancing': None, 'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'tree_method': 'hist', 'perangkat': 'cpu', 'weight_power': 0.5}
Ditimpa       : {}


## 2. Training 5 ulangan × 5 fold pasien

Bisa dilanjutkan. Bobot model, prediksi fold uji, dan prediksi uji stres tiap fold disimpan di `cache/v11/<model>/<NAMA_SETELAN>/`.

In [3]:
folder = runner.jalankan(MDL, NAMA_SETELAN, SETELAN)
print("Hasil:", folder)

Hasil: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11/xgboost/bawaan


## 3. Evaluasi (protokol v3)

Macro-F1 pooled OOF rata-rata ulangan + CI 95% bootstrap pasien. `+ bias` = bias kelas cross-fit (grid −7…+3). Pembanding v1 dan v3.1 memakai ulangan dan sampel bootstrap yang sama. Vonis: P ≥ 0,95 lebih baik, ≤ 0,05 lebih buruk, selebihnya tidak terbedakan.

In [4]:
r = evaluasi.laporan_model(folder, d)
label = {"v1": "v1 (argmax)", "v31": "v3.1 (referensi)", "argmax": f"{MODEL} argmax", "bias": f"{MODEL} + bias"}
fmt = lambda v: f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]"
kolom = ["Macro-F1", "AFL F1", "AFL P", "AFL R", "AFIB F1", "Normal R"]
print(f"Ulangan yang lengkap: {r['ulangan']}")
display(pd.DataFrame({label[k]: {m: fmt(r["metrik"][k][m]) for m in kolom} for k in label}).T)
display(pd.DataFrame({
    f"{MODEL} + bias vs {label[ref]}": {
        m: f"Δ {v[0]:+.3f} [{v[1]:+.2f}…{v[2]:+.2f}], P = {v[3]:.2f} → {evaluasi.vonis(v[3])}"
        for m, v in r[f"banding_{ref}"].items()}
    for ref in ("v31", "v1")}))

Ulangan yang lengkap: [0, 1, 2, 3, 4]


,Macro-F1,AFL F1,AFL P,AFL R,AFIB F1,Normal R
v1 (argmax),0.744 [0.64–0.81],0.399 [0.09–0.59],0.275 [0.05–0.46],0.728 [0.37–0.98],0.898 [0.85–0.93],0.903 [0.82–0.95]
v3.1 (referensi),0.832 [0.67–0.92],0.647 [0.17–0.89],0.715 [0.26–0.95],0.592 [0.13–0.91],0.909 [0.86–0.94],0.924 [0.84–0.97]
xgboost argmax,0.774 [0.65–0.81],0.478 [0.10–0.57],0.510 [0.07–0.67],0.453 [0.13–0.56],0.908 [0.87–0.94],0.924 [0.85–0.97]
xgboost + bias,0.730 [0.62–0.78],0.363 [0.04–0.51],0.394 [0.04–0.63],0.342 [0.05–0.53],0.895 [0.85–0.93],0.931 [0.86–0.97]


,xgboost + bias vs v3.1 (referensi),xgboost + bias vs v1 (argmax)
Macro-F1,"Δ -0.101 [-0.26…-0.02], P = 0.00 → lebih buruk","Δ -0.013 [-0.16…+0.05], P = 0.36 → tidak terbedakan"
AFL F1,"Δ -0.284 [-0.77…-0.04], P = 0.01 → lebih buruk","Δ -0.036 [-0.47…+0.15], P = 0.38 → tidak terbedakan"
AFL P,"Δ -0.321 [-0.73…-0.06], P = 0.00 → lebih buruk","Δ +0.119 [-0.22…+0.29], P = 0.77 → tidak terbedakan"
AFL R,"Δ -0.250 [-0.80…+0.04], P = 0.09 → tidak terbedakan","Δ -0.386 [-0.87…-0.14], P = 0.00 → lebih buruk"
Normal R,"Δ +0.007 [+0.00…+0.02], P = 0.99 → lebih baik","Δ +0.028 [+0.02…+0.04], P = 1.00 → lebih baik"


In [5]:
display(pd.DataFrame({label[k]: v for k, v in r["afl_per_pasien"].items()}).round(2).rename_axis("recall AFL per pasien"))
display(pd.DataFrame({label[k]: v for k, v in r["afl_palsu"].items()}).round(0).rename_axis("AFL palsu (window/ulangan)"))
b = r["bias"]
print(f"Bias [AFIB, AFL] median {b['median'][1:]}, rentang {b['min'][1:]} … {b['maks'][1:]}")
if b["di_batas"] > 0:
    print(f"⚠ {b['di_batas']:.0%} bias menyentuh batas grid (−7/+3): tanda bahaya (AGENTS.md, pelajaran v2/v9).")

,v1 (argmax),v3.1 (referensi),xgboost argmax,xgboost + bias
recall AFL per pasien,,,,
04043,1.00,0.40,0.20,0.20
04908,1.00,0.92,1.00,1.00
04936,0.67,0.51,0.52,0.49
06426,0.73,0.17,0.07,0.08
07910,0.21,0.00,0.03,0.04
08215,0.04,0.00,0.00,0.00
08378,1.00,0.96,0.39,0.05


,v1 (argmax),v3.1 (referensi),xgboost argmax,xgboost + bias
AFL palsu (window/ulangan),,,,
total,2240.0,274.0,501.0,600.0
dari_normal,521.0,55.0,210.0,217.0
dari_afib,1719.0,219.0,291.0,383.0
pasien_tanpa_afl,1059.0,25.0,149.0,252.0


Bias [AFIB, AFL] median [-0.75, 0.0], rentang [-1.5, -6.0] … [1.25, 1.75]


## 4. Uji stres takikardia (protokol butir 9)

Ritme teratur sintetis 60–170 bpm × 4 jenis variasi detak. **Ambang** = detak terendah di mana < 50% window ditebak Normal (makin tinggi makin baik). Pembanding: kontrol MicroTransformer per fold dari v7 dan paket produksi v3.1 (115 bpm).

In [6]:
tk = r["takikardia"]
display(pd.DataFrame({MODEL: tk["model"]["ambang"], "kontrol v7 (per fold)": tk["kontrol_v7"]["ambang"],
                      "paket v3.1": {j: tk["paket_v31"] for j in tk["model"]["ambang"]}}).rename_axis("ambang (bpm)"))
normal = pd.DataFrame({j: [p[0] for p in tab] for j, tab in tk["model"]["proporsi"].items()}, index=tk["model"]["bpm"])
display((normal * 100).round(0).T.rename_axis(f"% ditebak Normal oleh {MODEL}"))
pita = {k: {"window": v["n"], "pasien": v["n_pasien"],
            "v3.1: % AFIB/AFL": 100 * (v["v31"][1] + v["v31"][2]),
            f"{MODEL}: % AFIB/AFL": 100 * (v["model"][1] + v["model"][2])}
        for k, v in r["pita_detak"].items() if k.startswith("Normal")}
display(pd.DataFrame(pita).T.round(1).rename_axis("window Normal asli per pita detak"))

,xgboost,kontrol v7 (per fold),paket v3.1
ambang (bpm),,,
tanpa variasi,125,120,115
seragam ±10 ms,120,120,115
Gauss sd 10 ms,115,120,115
Gauss sd 20 ms,115,115,115


,60,65,70,75,80,85,90,95,100,105,...,125,130,135,140,145,150,155,160,165,170
% ditebak Normal oleh xgboost,,,,,,,,,,,,,,,,,,,,,
tanpa variasi,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,...,28.0,12.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
seragam ±10 ms,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,...,9.0,2.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
Gauss sd 10 ms,100.0,100.0,100.0,100.0,100.0,83.0,86.0,100.0,100.0,100.0,...,28.0,3.0,2.0,1.0,1.0,0.0,0.0,1.0,2.0,3.0
Gauss sd 20 ms,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,98.0,...,16.0,1.0,1.0,1.0,0.0,0.0,0.0,2.0,3.0,3.0


,window,pasien,v3.1: % AFIB/AFL,xgboost: % AFIB/AFL
window Normal asli per pita detak,,,,
Normal | 0–100 bpm,89566.0,21.0,6.7,6.0
Normal | 100–110 bpm,8766.0,12.0,12.7,12.0
Normal | 110–120 bpm,568.0,10.0,59.0,56.0
Normal | 120–∞ bpm,100.0,7.0,95.2,85.0


## 5. Kesimpulan

Setelan `bawaan`, 5 ulangan lengkap. Training di PC (CPU, 2 s per fold) lewat `bersama.runner` (23 Oktober 2026); notebook ini dijalankan di PC yang sama dan hanya mengevaluasi fold yang sudah ada (3 November 2026). Rincian: `README.md`.

- **Macro-F1 + bias 0,730 [0,62–0,78]: lebih buruk daripada v3.1** (Δ −0,101, P < 0,01) dan BiGRU bawaan (−0,091, P < 0,001), setara v1 (−0,013, P = 0,36). Bukan kandidat pengganti paket produksi.
- **Bias merugikan XGBoost:** tanpa bias Macro-F1 0,774. Bias AFL melompat antar fold (−6 … +1,75, median 0), dan AFL palsu justru naik (501 → 600 window per ulangan, 2,2× v3.1). Pola yang sama dengan LightGBM v3.
- **AFL F1 0,363 [0,04–0,51]**, recall AFL 0,342 [0,05–0,53]. AFL 2:1 (08378) hampir tidak terdeteksi (0,05; tanpa bias 0,39).
- **Recall AFL per pasien (+ bias):** 04043 0,20 · 04908 1,00 · 04936 0,49 · 06426 0,08 · 07910 0,04 · 08215 0,00 · 08378 0,05.
- **Takikardia:** ambang 125 / 120 / 115 / 115 bpm (v3.1 115). Alarm pada Normal ≥ 120 bpm 85,0% (v3.1 95,2%).
- **Balancing dan bobot** (23 Oktober 2026): `bobot_penuh` dan `smote` sama-sama +0,013 vs bawaan (P = 0,97 dan 0,98) dan tidak terbedakan satu sama lain. `under` 0,794, +0,063 vs bawaan (P = 0,98), tetapi tetap lebih buruk daripada v3.1 (P < 0,001). Seleksi cross-fit `bawaan`/`bobot_penuh`: 0,745. Lihat `README.md`.
- Hanya 16 fitur HRV; hiperparameter pohon belum di-tuning.

In [7]:
print(f"Latih rata-rata {r['waktu_per_fold']:.0f} s per fold | info model: {r['info']}")
print("Zip hasil untuk dibagikan ke tim:", runner.zip_hasil(folder))

Latih rata-rata 2 s per fold | info model: {'pohon': 400, 'n_latih': [82012, 46765, 1143]}


Zip hasil untuk dibagikan ke tim: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11/xgboost/xgboost_bawaan.zip
